# 02 — Glue: Data Catalog Management

AWS Glue Data Catalog is the central metadata repository for your data lake. Athena, EMR, and Spark all read table definitions from here.

- Section A: Database & Table CRUD
- Section B: StorageDescriptor deep-dive (the most important dict in Glue)
- Section C: Partition management
- Section D: Crawlers (conceptual + mock)
- Section E: ETL Job lifecycle

In [1]:
import sys, json
sys.path.insert(0, "..")
import helpers

import boto3
import pandas as pd
from moto import mock_aws

---
## Section A — Database & Table CRUD

In [2]:
# A1: Create and list databases
with mock_aws():
    glue = helpers.make_boto3_client("glue")

    glue.create_database(DatabaseInput={"Name": "analytics_db", "Description": "Main analytics database"})
    glue.create_database(DatabaseInput={"Name": "raw_db"})

    dbs = glue.get_databases()["DatabaseList"]
    print("Databases:", [d["Name"] for d in dbs])

    detail = glue.get_database(Name="analytics_db")["Database"]
    print("Description:", detail.get("Description"))

Databases: ['analytics_db', 'raw_db']
Description: Main analytics database


---
## Section B — StorageDescriptor (the critical struct)

The `StorageDescriptor` tells Glue (and Athena) where the data lives and how to read it. **Interviewers will ask you to construct this from scratch.** Learn every field.

In [3]:
# B1: Complete create_table with a real StorageDescriptor
with mock_aws():
    glue = helpers.make_boto3_client("glue")
    glue.create_database(DatabaseInput={"Name": "demo_db"})

    glue.create_table(
        DatabaseName="demo_db",
        TableInput={
            "Name": "orders",
            "Description": "Raw orders from OLTP system",
            # Non-partition columns
            "StorageDescriptor": {
                # Where the data files live in S3
                "Location": "s3://my-datalake/raw/orders/",
                # Columns (separate from partition columns)
                "Columns": [
                    {"Name": "order_id",   "Type": "bigint"},
                    {"Name": "customer_id","Type": "bigint"},
                    {"Name": "amount",     "Type": "double"},
                    {"Name": "status",     "Type": "string"},
                ],
                # How to read the file — CSV:
                "InputFormat":  "org.apache.hadoop.mapred.TextInputFormat",
                "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
                # SerDe = Serializer/Deserializer — tells Glue how to parse the bytes
                "SerdeInfo": {
                    "SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe",
                    "Parameters": {
                        "field.delim": ",",         # CSV delimiter
                        "skip.header.line.count": "1",  # skip the header row
                    },
                },
                "Compressed": False,
            },
            # Partition columns — these are the year/month/day keys
            "PartitionKeys": [
                {"Name": "year",  "Type": "int"},
                {"Name": "month", "Type": "int"},
            ],
            "TableType": "EXTERNAL_TABLE",
            "Parameters": {"classification": "csv"},
        },
    )

    table = glue.get_table(DatabaseName="demo_db", Name="orders")["Table"]
    print("Table name:", table["Name"])
    print("Location:", table["StorageDescriptor"]["Location"])
    col_names = [c["Name"] for c in table["StorageDescriptor"]["Columns"]]
    print("Columns:", col_names)
    print("Partition keys:", [p["Name"] for p in table["PartitionKeys"]])

Table name: orders
Location: s3://my-datalake/raw/orders/
Columns: ['order_id', 'customer_id', 'amount', 'status']
Partition keys: ['year', 'month']


In [4]:
# B2: StorageDescriptor for Parquet (different InputFormat + SerDe)
PARQUET_STORAGE_DESCRIPTOR = {
    "Location": "s3://my-datalake/processed/events/",
    "Columns": [
        {"Name": "event_id", "Type": "string"},
        {"Name": "user_id",  "Type": "bigint"},
        {"Name": "ts",       "Type": "timestamp"},
    ],
    "InputFormat":  "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
    "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
    "SerdeInfo": {
        "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
        "Parameters": {"serialization.format": "1"},
    },
    "Compressed": True,
}
print("Parquet SerDe:", PARQUET_STORAGE_DESCRIPTOR["SerdeInfo"]["SerializationLibrary"])

Parquet SerDe: org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe


In [5]:
# B3: Update a table (add a new column)
with mock_aws():
    glue = helpers.make_boto3_client("glue")
    glue.create_database(DatabaseInput={"Name": "update_db"})
    glue.create_table(
        DatabaseName="update_db",
        TableInput={
            "Name": "users",
            "StorageDescriptor": {
                "Location": "s3://bucket/users/",
                "Columns": [{"Name": "user_id", "Type": "bigint"}],
                "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
                "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
                "SerdeInfo": {"SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe"},
            },
            "PartitionKeys": [],
        },
    )

    # Add a new column via update_table
    existing = glue.get_table(DatabaseName="update_db", Name="users")["Table"]
    updated_columns = existing["StorageDescriptor"]["Columns"] + [{"Name": "email", "Type": "string"}]

    glue.update_table(
        DatabaseName="update_db",
        TableInput={
            "Name": "users",
            "StorageDescriptor": {
                **existing["StorageDescriptor"],
                "Columns": updated_columns,
            },
            "PartitionKeys": existing["PartitionKeys"],
        },
    )

    cols = glue.get_table(DatabaseName="update_db", Name="users")["Table"]["StorageDescriptor"]["Columns"]
    print("Columns after update:", [c["Name"] for c in cols])

Columns after update: ['user_id', 'email']


---
## Section C — Partition Management

In [6]:
# C1: Helper to build a PartitionInput dict
def make_partition(location: str, values: list[str]) -> dict:
    """Build a PartitionInput dict for glue.create_partition / batch_create_partition."""
    return {
        "Values": values,
        "StorageDescriptor": {
            "Location": location,
            "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
            "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
            "SerdeInfo": {"SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe"},
            "Columns": [],  # inherits from table definition
        },
    }

print("Helper defined.")

Helper defined.


In [7]:
# C2: Add a single partition
with mock_aws():
    glue = helpers.make_boto3_client("glue")
    glue.create_database(DatabaseInput={"Name": "part_db"})
    glue.create_table(
        DatabaseName="part_db",
        TableInput={
            "Name": "logs",
            "StorageDescriptor": {
                "Location": "s3://datalake/logs/",
                "Columns": [{"Name": "message", "Type": "string"}],
                "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
                "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
                "SerdeInfo": {"SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe"},
            },
            "PartitionKeys": [{"Name": "dt", "Type": "string"}],
        },
    )

    # Single partition
    glue.create_partition(
        DatabaseName="part_db",
        TableName="logs",
        PartitionInput=make_partition("s3://datalake/logs/dt=2024-01-01/", ["2024-01-01"]),
    )

    parts = glue.get_partitions(DatabaseName="part_db", TableName="logs")["Partitions"]
    print("Partitions:", [p["Values"] for p in parts])

Partitions: [['2024-01-01']]


In [8]:
# C3: batch_create_partition — add multiple partitions in one API call
with mock_aws():
    glue = helpers.make_boto3_client("glue")
    glue.create_database(DatabaseInput={"Name": "batch_db"})
    glue.create_table(
        DatabaseName="batch_db",
        TableInput={
            "Name": "events",
            "StorageDescriptor": {
                "Location": "s3://bucket/events/",
                "Columns": [{"Name": "payload", "Type": "string"}],
                "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
                "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
                "SerdeInfo": {"SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe"},
            },
            "PartitionKeys": [{"Name": "dt", "Type": "string"}],
        },
    )

    dates = ["2024-01-01", "2024-01-02", "2024-01-03"]
    partition_inputs = [
        make_partition(f"s3://bucket/events/dt={d}/", [d])
        for d in dates
    ]

    resp = glue.batch_create_partition(
        DatabaseName="batch_db",
        TableName="events",
        PartitionInputList=partition_inputs,
    )
    print("Errors (should be empty):", resp.get("Errors", []))

    parts = glue.get_partitions(DatabaseName="batch_db", TableName="events")["Partitions"]
    print("Partitions created:", [p["Values"][0] for p in parts])

Errors (should be empty): []
Partitions created: ['2024-01-01', '2024-01-02', '2024-01-03']


---
## Section D — Crawlers (Conceptual + Mock)

A Glue Crawler scans S3, infers the schema, and writes table definitions to the catalog automatically.

**moto limitation:** Crawlers in moto do NOT scan S3 or create tables. Use this section to learn the API shape; use `create_table` to simulate what a crawler would produce.

In [9]:
# D1: Create a crawler (API practice — moto does not execute it)
with mock_aws():
    glue = helpers.make_boto3_client("glue")
    glue.create_database(DatabaseInput={"Name": "crawler_db"})

    glue.create_crawler(
        Name="raw-data-crawler",
        Role=helpers.DUMMY_ROLE_ARN,
        DatabaseName="crawler_db",
        Targets={
            "S3Targets": [{"Path": "s3://my-datalake/raw/"}]
        },
        Schedule="cron(0 6 * * ? *)",  # daily at 6am UTC
    )

    crawler = glue.get_crawler(Name="raw-data-crawler")["Crawler"]
    print("Crawler name:", crawler["Name"])
    print("Target:", crawler["Targets"]["S3Targets"][0]["Path"])
    print("State:", crawler["State"])  # READY

Crawler name: raw-data-crawler
Target: s3://my-datalake/raw/
State: READY


In [10]:
# D2: Start and stop a crawler
with mock_aws():
    glue = helpers.make_boto3_client("glue")
    glue.create_database(DatabaseInput={"Name": "crawler_db"})
    glue.create_crawler(
        Name="my-crawler",
        Role=helpers.DUMMY_ROLE_ARN,
        DatabaseName="crawler_db",
        Targets={"S3Targets": [{"Path": "s3://bucket/raw/"}]},
    )

    glue.start_crawler(Name="my-crawler")
    state = glue.get_crawler(Name="my-crawler")["Crawler"]["State"]
    print("State after start:", state)  # RUNNING

    glue.stop_crawler(Name="my-crawler")
    state = glue.get_crawler(Name="my-crawler")["Crawler"]["State"]
    print("State after stop:", state)  # STOPPING

State after start: RUNNING
State after stop: STOPPING


---
## Section E — ETL Job Lifecycle

Glue ETL jobs run PySpark code on a managed cluster. In moto, jobs are created and runs are tracked — but no actual Spark code executes.

In [11]:
# E1: Create a Glue ETL job
with mock_aws():
    glue = helpers.make_boto3_client("glue")

    glue.create_job(
        Name="transform-orders",
        Role=helpers.DUMMY_ROLE_ARN,
        Command={
            "Name": "glueetl",
            "ScriptLocation": "s3://my-scripts/transform_orders.py",
            "PythonVersion": "3",
        },
        DefaultArguments={
            "--job-language": "python",
            "--TempDir": "s3://my-temp/",
            "--enable-metrics": "",
        },
        GlueVersion="4.0",
        NumberOfWorkers=5,
        WorkerType="G.1X",
    )

    jobs = glue.list_jobs()["JobNames"]
    print("Jobs:", jobs)

Jobs: ['transform-orders']


In [12]:
# E2: Start a job run and poll its status
import time

with mock_aws():
    glue = helpers.make_boto3_client("glue")
    glue.create_job(
        Name="my-etl-job",
        Role=helpers.DUMMY_ROLE_ARN,
        Command={"Name": "glueetl", "ScriptLocation": "s3://bucket/script.py"},
        DefaultArguments={},
    )

    run = glue.start_job_run(
        JobName="my-etl-job",
        Arguments={"--date": "2024-01-15"}
    )
    run_id = run["JobRunId"]
    print("Started run:", run_id)

    # Poll until not RUNNING (in moto this completes immediately)
    max_polls = 5
    for attempt in range(max_polls):
        status = glue.get_job_run(JobName="my-etl-job", RunId=run_id)["JobRun"]["JobRunState"]
        print(f"  Poll {attempt+1}: {status}")
        if status not in ("RUNNING", "STARTING", "STOPPING"):
            break
        time.sleep(1)

    # List all runs
    all_runs = glue.get_job_runs(JobName="my-etl-job")["JobRuns"]
    print(f"Total runs: {len(all_runs)}, final state: {all_runs[0]['JobRunState']}")

Started run: jr_2e611afca5180dd8ea5ebb19ae4a95f34bdd90a23f10cb7f08169eedfd464116
  Poll 1: SUCCEEDED
Total runs: 1, final state: SUCCEEDED


## Summary

| API Call | Purpose |
|---|---|
| `create_database` | Create a catalog database |
| `create_table` | Register table with StorageDescriptor |
| `update_table` | Modify schema (add/rename columns) |
| `get_table` / `get_tables` | Read table metadata |
| `create_partition` | Register one partition |
| `batch_create_partition` | Register many partitions in one call |
| `get_partitions` | List all partitions for a table |
| `create_crawler` / `start_crawler` | Define + trigger S3 scan (moto: API only) |
| `create_job` / `start_job_run` | Define + trigger ETL job |
| `get_job_run` | Poll job run status |

**StorageDescriptor cheat sheet:**
- CSV: `LazySimpleSerDe` + `TextInputFormat`
- Parquet: `ParquetHiveSerDe` + `MapredParquetInputFormat`
- JSON: `JsonSerDe` + `TextInputFormat`

**Next notebook:** [03_athena_serverless_query_pipeline.ipynb](03_athena_serverless_query_pipeline.ipynb)